# Fraught corners and λ₂ on 7-node graphs

For each graph (Full, Star, Hierarchy, CT-Hierarchy, Ring, Path) we

1. write the edge list,
2. build a NetworkX graph,
3. form the Laplacian `L = D - A` and compute λ₂,
4. enumerate all 2⁷ = 128 corners `s ∈ {+1, -1}⁷` and count the fraught ones.

A corner `s` is **fraught** if `s_i * (A s)_i > 0` for every node `i`.

In [1]:
import itertools
import numpy as np
import networkx as nx
import pandas as pd
from slider.fraught import is_fraught

n = 7
nodes = list(range(1, n + 1))

## 1. Edge lists

Hierarchy is the binary tree 1 → (2, 3), 2 → (4, 5), 3 → (6, 7).

**Assumption:** CT-Hierarchy is taken as the Hierarchy plus cross-team links 2–3 and 5–6. Edit the list below if your definition is different.

In [2]:
hierarchy = [(1, 2), (1, 3), (2, 4), (2, 5), (3, 6), (3, 7)]

edges = {
    "Full":         [(i, j) for i in nodes for j in nodes if i < j],
    "Star":         [(1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7)],
    "Hierarchy":    hierarchy,
    "CT-Hierarchy": hierarchy + [(4, 5), (6, 7)],
    "Ring":         [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 1)],
    "Path":         [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)],
}

for name, e in edges.items():
    print(name, len(e), "edges:", e)

Full 21 edges: [(1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 4), (3, 5), (3, 6), (3, 7), (4, 5), (4, 6), (4, 7), (5, 6), (5, 7), (6, 7)]
Star 6 edges: [(1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7)]
Hierarchy 6 edges: [(1, 2), (1, 3), (2, 4), (2, 5), (3, 6), (3, 7)]
CT-Hierarchy 8 edges: [(1, 2), (1, 3), (2, 4), (2, 5), (3, 6), (3, 7), (4, 5), (6, 7)]
Ring 7 edges: [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 1)]
Path 6 edges: [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)]


## 2. NetworkX graphs

In [3]:
graphs = {}
for name, e in edges.items():
    G = nx.Graph()
    G.add_nodes_from(nodes)
    G.add_edges_from(e)
    graphs[name] = G

## 3. Laplacian `L = D - A` and λ₂

`A` comes from NetworkX, then `L = D - A` is done in NumPy so the formula is visible.
(`nx.laplacian_matrix(G)` gives the same matrix.)

λ₂ is the second-smallest eigenvalue of `L`.

In [4]:
A = {}
L = {}
lam2 = {}
for name, G in graphs.items():
    A[name] = nx.to_numpy_array(G, nodelist=nodes)
    D = np.diag(A[name].sum(axis=1))
    L[name] = D - A[name]
    lam2[name] = np.linalg.eigvalsh(L[name])[1]   # eigenvalues are sorted ascending

for name in graphs:
    print(name)
    print(L[name].astype(int))
    print("lambda_2 =", round(lam2[name], 4))
    print()

Full
[[ 6 -1 -1 -1 -1 -1 -1]
 [-1  6 -1 -1 -1 -1 -1]
 [-1 -1  6 -1 -1 -1 -1]
 [-1 -1 -1  6 -1 -1 -1]
 [-1 -1 -1 -1  6 -1 -1]
 [-1 -1 -1 -1 -1  6 -1]
 [-1 -1 -1 -1 -1 -1  6]]
lambda_2 = 7.0

Star
[[ 6 -1 -1 -1 -1 -1 -1]
 [-1  1  0  0  0  0  0]
 [-1  0  1  0  0  0  0]
 [-1  0  0  1  0  0  0]
 [-1  0  0  0  1  0  0]
 [-1  0  0  0  0  1  0]
 [-1  0  0  0  0  0  1]]
lambda_2 = 1.0

Hierarchy
[[ 2 -1 -1  0  0  0  0]
 [-1  3  0 -1 -1  0  0]
 [-1  0  3  0  0 -1 -1]
 [ 0 -1  0  1  0  0  0]
 [ 0 -1  0  0  1  0  0]
 [ 0  0 -1  0  0  1  0]
 [ 0  0 -1  0  0  0  1]]
lambda_2 = 0.2679

CT-Hierarchy
[[ 2 -1 -1  0  0  0  0]
 [-1  3  0 -1 -1  0  0]
 [-1  0  3  0  0 -1 -1]
 [ 0 -1  0  2 -1  0  0]
 [ 0 -1  0 -1  2  0  0]
 [ 0  0 -1  0  0  2 -1]
 [ 0  0 -1  0  0 -1  2]]
lambda_2 = 0.2679

Ring
[[ 2 -1  0  0  0  0 -1]
 [-1  2 -1  0  0  0  0]
 [ 0 -1  2 -1  0  0  0]
 [ 0  0 -1  2 -1  0  0]
 [ 0  0  0 -1  2 -1  0]
 [ 0  0  0  0 -1  2 -1]
 [-1  0  0  0  0 -1  2]]
lambda_2 = 0.753

Path
[[ 1 -1  0  0  0  0  0]


## 5. Fraught corners

For a corner `s`, compute `s_i * (A s)_i` for each `i`. The corner is fraught if all 7 values are positive.

In [5]:
corners = np.array(list(itertools.product([1, -1], repeat=n)))
corners = corners[abs(corners.sum(axis=1)) != n]   # remove all +1 and all -1
print(corners.shape)   # (126, 7)

fraught = {}
for name in graphs:
    fraught[name] = [is_fraught(A[name], s) for s in corners]

(126, 7)


## 6. Summary table

In [6]:
summary = pd.DataFrame({
    "edges": [len(edges[name]) for name in graphs],
    "fraught corners": [sum(fraught[name]) for name in graphs],
    "lambda_2": [round(lam2[name], 4) for name in graphs],
}, index=list(graphs))
summary

,edges,fraught corners,lambda_2
Full,21,0,7.0000
Star,6,0,1.0000
Hierarchy,6,4,0.2679
CT-Hierarchy,8,4,0.2679
Ring,7,28,0.7530
Path,6,14,0.1981


## 7. Fraught fraction in the cube [−1, 1]⁷ (Monte Carlo)

Sample `N = 10⁶` points `s` uniformly from [−1, 1]⁷.

A point is fraught if `s_i * (A s)_i >= 0` for every `i` **and** it is not in the orthant O₊ (all `s_i > 0`) or O₋ (all `s_i < 0`).

The fraction `p` is out of all `N` points, and `SE = √(p(1 − p)/N)`.

Target for Hierarchy: 5/384 ≈ 1.302%.

In [7]:
N = 10**6
rng = np.random.default_rng(0)
S = rng.uniform(-1, 1, size=(N, n))   # one sample per row

in_orthant = np.all(S > 0, axis=1) | np.all(S < 0, axis=1)   # O+ or O-

rows = []
for name in graphs:
    ok = np.all(S * (S @ A[name]) >= 0, axis=1)   # row k of S @ A is A @ s_k (A is symmetric)
    p = np.mean(ok & ~in_orthant)
    se = np.sqrt(p * (1 - p) / N)
    rows.append([p, se])

mc = pd.DataFrame(rows, columns=["fraught fraction", "SE"], index=list(graphs))
mc

,fraught fraction,SE
Full,0.000000,0.000000
Star,0.000000,0.000000
Hierarchy,0.012907,0.000113
CT-Hierarchy,0.013645,0.000116
Ring,0.012201,0.000110
Path,0.017584,0.000131
